# 🗓 Calendar Agent — Kaggle LoRA Training

Colab 대안. Kaggle은 별도 GPU quota를 제공 (주 30h, T4×2 또는 P100 16GB).

## 실행 전 준비 (한 번만)

### 1. Kaggle 설정 (우측 패널)
- **Accelerator**: `GPU T4 x2` 또는 `GPU P100` 선택
- **Internet**: `On` (외부 패키지·HF 모델 다운로드용)
- **Persistence**: `Variables and Files` (선택 — 세션 재시작 시 일부 보존)

### 2. 데이터 업로드 (Kaggle Dataset으로)
1. https://www.kaggle.com/datasets → `New Dataset`
2. 3개 파일 업로드:
   - `v1_train.jsonl` (로컬 `D:\calendar-agent\data\processed\v1_train.jsonl`)
   - `v1_val.jsonl` (동)
   - `golden.jsonl` (로컬 `D:\calendar-agent\data\eval\golden.jsonl`)
3. Dataset 이름: `calendar-agent-data` (또는 임의)
4. Visibility: **Private**
5. Create

### 3. 이 노트북에 Dataset 첨부
- 우측 패널 `+ Add Input` → `Datasets` 탭 → 위에서 만든 dataset 검색 → 첨부
- 첨부되면 `/kaggle/input/calendar-agent-data/` 경로에 파일 보임

### 4. GitHub PAT 발급 (Private repo clone용)
https://github.com/settings/tokens/new?type=classic — repo scope, 7 days

---

준비 끝나면 셀을 위에서 아래로 순서대로 실행. 예상 시간: ~1시간.

## 0. GPU 확인

`Tesla T4` 또는 `Tesla P100` 표시되어야 함. 안 보이면 우측 패널 Accelerator 다시.

In [ ]:
!nvidia-smi

## 1. 레포 clone (Private)

Kaggle은 working dir이 `/kaggle/working/`. 그 안에 clone.

In [ ]:
import os, getpass

%cd /kaggle/working
if not os.path.exists('calendar-agent'):
    token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    !git clone https://{token}@github.com/soo-vibe/calendar-agent.git
    token = None
else:
    print('calendar-agent already cloned, pulling latest…')
    !cd calendar-agent && git pull

%cd /kaggle/working/calendar-agent
!git log --oneline -1

## 2. 학습 의존성 설치 + Colab/Kaggle 호환 cleanup

Colab과 동일 — install + torchao 제거.

In [ ]:
!pip install -q -e .[train]
!pip uninstall torchao -y
import torch
print(f'\ntorch={torch.__version__}  cuda={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## 3. 데이터 복사 (Kaggle Dataset → 작업 디렉토리)

첨부한 Dataset이 `/kaggle/input/calendar-agent-data/`에 마운트됨.
작업 디렉토리에 repo 구조대로 복사.

**참고**: Dataset 이름이 다르면 `DATASET_DIR`만 수정.

In [ ]:
import shutil
from pathlib import Path

DATASET_DIR = Path('/kaggle/input/calendar-agent-data')
TRAIN = Path('data/processed/v1_train.jsonl')
VAL   = Path('data/processed/v1_val.jsonl')
GOLD  = Path('data/eval/golden.jsonl')
TRAIN.parent.mkdir(parents=True, exist_ok=True)
GOLD.parent.mkdir(parents=True, exist_ok=True)

for fn, dst in [('v1_train.jsonl', TRAIN), ('v1_val.jsonl', VAL), ('golden.jsonl', GOLD)]:
    src = DATASET_DIR / fn
    if not src.exists():
        raise FileNotFoundError(f'{src} not found. Dataset 첨부했는지 / 파일 이름 일치하는지 확인.')
    if not dst.exists():
        shutil.copy(src, dst)
        print(f'{src} → {dst}')
    else:
        print(f'{dst} already exists, skipping copy.')

print()
for p in [TRAIN, VAL, GOLD]:
    print(f'✓ {p}' if p.exists() else f'✗ MISSING: {p}')

## 4. Sanity check + config 자동 조정

- 3개 파일 shape 검증
- WandB 끄기
- bf16 미지원 GPU(T4)면 fp16로 전환

In [ ]:
import orjson, torch
from pathlib import Path

for p in [Path('data/processed/v1_train.jsonl'), Path('data/processed/v1_val.jsonl'), Path('data/eval/golden.jsonl')]:
    assert p.exists(), f'MISSING: {p}'
    n = sum(1 for line in open(p, 'rb') if line.strip())
    print(f'{p}: {n} records')

# wandb off
!sed -i 's/report_to: wandb/report_to: none/' configs/train.yaml

# bf16 -> fp16 if needed
if not torch.cuda.is_bf16_supported():
    !sed -i 's/bf16: true/bf16: false/' configs/train.yaml
    !sed -i 's/fp16: false/fp16: true/' configs/train.yaml
    !sed -i "s|bnb_4bit_compute_dtype: bfloat16|bnb_4bit_compute_dtype: float16|" configs/train.yaml
    print('GPU에 bf16 미지원 — fp16로 전환')
else:
    print('bf16 사용 (P100/A100)')

!grep -E 'report_to|bf16|fp16|bnb_4bit_compute' configs/train.yaml

## 5. 학습 실행

1970건 × 3 epochs ≈ 165 step. T4/P100 기준 ~1시간.

**Kaggle 세션 한도**: 9시간/세션 — 우리 학습은 1시간이라 무관.

In [ ]:
!python scripts/train_lora.py --config configs/train.yaml

## 6. 결과 패키징 (Kaggle Output 자동 노출)

Kaggle은 `/kaggle/working/` 안의 파일을 노트북 우측 패널 `Output`에서 다운로드 가능.
zip을 `/kaggle/working/` 루트에 두면 됨.

In [ ]:
!ls -la models/lora/v1/
!zip -r /kaggle/working/lora_v3.zip models/lora/v1 configs/train.yaml configs/lora.yaml configs/model_qwen.yaml
!ls -la /kaggle/working/lora_v3.zip

from IPython.display import FileLink
FileLink('/kaggle/working/lora_v3.zip')

## 다운로드 방법

1. **위 셀의 FileLink 클릭** — 바로 다운로드
2. **또는 우측 패널 `Output` 탭** — `lora_v3.zip` 옆 `...` → `Download`
3. **Kaggle 노트북 commit** (옵션) — 위 결과를 Kaggle에 저장해서 다음에 재사용 가능. 우측 상단 `Save Version` → `Quick Save` 또는 `Save & Run All`.